## Construir el modelo de transformador con PyTorch
Para construir el modelo de Transformador, son necesarios los siguientes pasos:

1. Importar las bibliotecas y módulos.
2. Definir los elementos básicos: Atención multicabezal, redes de alimentación en función de la posición, codificación posicional.
3. Construir el bloque Codificador.
4. Construir el bloque Decodificador.
5. Combinar las capas Codificador y Decodificador para crear la red Transformador completa.

### 1. Importar las bibliotecas y módulos necesarios
Empezaremos importando la biblioteca PyTorch para la funcionalidad básica, el módulo de redes neuronales para crear redes neuronales, el módulo de optimización para entrenar redes y las funciones de utilidad de datos para manejar datos. Además, importaremos el módulo estándar de Python math para operaciones matemáticas y el módulo copy para crear copias de objetos complejos.

Estas herramientas sientan las bases para definir la arquitectura del modelo, gestionar los datos y establecer el proceso de entrenamiento.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy


### 2. Definir los elementos básicos: Atención multicabezal, redes de alimentación en función de la posición, codificación posicional
Antes de empezar a construir nuestros componentes, echa un vistazo a la siguiente tabla, que describe los distintos componentes de un Transformador y su finalidad: 

| Componente               | Descripción                                          | Propósito                                                  |
|--------------------------|------------------------------------------------------|------------------------------------------------------------|
| Atención multicabeza     | Mecanismo para centrarse en diferentes partes de la entrada | Capta las dependencias en diferentes posiciones de la secuencia |
| Redes Feed-Forward       | Capas totalmente conectadas en función de la posición | Transforma las salidas de atención, añadiendo complejidad  |
| Codificación posicional  | Añade información posicional a las incrustaciones    | Proporciona contexto de orden secuencial al modelo        |
| Normalización de capas   | Normaliza las entradas de cada subcapa               | Estabiliza el entrenamiento, mejora la convergencia       |
| Conexiones residuales    | Atajos entre capas                                   | Ayuda a entrenar redes más profundas minimizando los problemas de gradiente |
| Abandono                 | Pone a cero aleatoriamente algunas conexiones de red | Evita el sobreajuste regularizando el modelo              |

#### Atención multicabezal
El mecanismo de atención multicabezal calcula la atención entre cada par de posiciones de una secuencia. Consta de múltiples "cabezas de atención" que captan diferentes aspectos de la secuencia de entrada.


In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        # Initialize dimensions
        self.d_model = d_model  # Model's dimension
        self.num_heads = num_heads  # Number of attention heads
        self.d_k = (
            d_model // num_heads
        )  # Dimension of each head's key, query, and value

        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model)  # Query transformation
        self.W_k = nn.Linear(d_model, d_model)  # Key transformation
        self.W_v = nn.Linear(d_model, d_model)  # Value transformation
        self.W_o = nn.Linear(d_model, d_model)  # Output transformation

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)

        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output


La clase se define como una subclase de nn.Module de PyTorch.

1. d_model: Dimensionalidad de la entrada.
2. num_heads: El número de cabezas de atención en que dividir la entrada.

La inicialización comprueba si d_model es divisible por num_heads, y luego define los pesos de transformación para query, key, value y output.

##### Atención al producto punto escalado:


La atención al producto punto escalado es un mecanismo que calcula la atención entre las posiciones de la secuencia. Se basa en el producto punto entre las matrices de consulta (Q) y clave (K), seguido de una normalización por la raíz cuadrada de la dimensión de la clave (d_k). Esto ayuda a estabilizar los gradientes durante el entrenamiento.
La función de atención toma las matrices de consulta, clave y valor, calcula la atención y devuelve la salida. La atención se calcula como el producto punto entre Q y K, seguido de una normalización softmax. Luego, se multiplica por V para obtener la salida final.

```python
def scaled_dot_product_attention(self, Q, K, V, mask=None):
```

1. Cálculo de las puntuaciones de atención: attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k). Aquí, las puntuaciones de atención se calculan tomando el producto punto de las consultas (Q) y las claves (K), y luego escalando por la raíz cuadrada de la dimensión de la clave (d_k).
2. Aplicar la máscara: Si se proporciona una máscara, se aplica a las puntuaciones de atención para enmascarar valores específicos.
3. Calcular los pesos de atención: Las puntuaciones de atención se pasan por una función softmax para convertirlas en probabilidades que sumen 1.
4. Calculando la salida: El resultado final de la atención se calcula multiplicando los pesos de la atención por los valores (V).

##### Partiendo cabezas:
```python
def scaled_dot_product_attention(self, Q, K, V, mask=None):
```
Este método remodela la entrada x en la forma (batch_size, num_heads, seq_length, d_k). Permite al modelo procesar varias cabezas de atención simultáneamente, lo que permite el cálculo paralelo.

##### Combinar cabezas:

```python
def combine_heads(self, x):
```
Tras aplicar la atención a cada cabeza por separado, este método vuelve a combinar los resultados en un único tensor de forma (batch_size, seq_length, d_model). Esto prepara el resultado para su posterior procesamiento en la red neuronal.

##### Método de avance (forward):
```python
def forward(self, Q, K, V, mask=None):
```

El método directo es donde se produce el cálculo real:

1. Aplica transformaciones lineales: Las consultas (Q), las claves (K) y los valores (V) se someten primero a transformaciones lineales 2. utilizando los pesos definidos en la inicialización.
2. Cabezas partidas: Las transformadas Q, K, V se dividen en varias cabezas mediante el método split_heads.
3. Aplica la atención del producto punto escalado: Se llama al método scaled_dot_product_attention en las cabezas divididas.
4. Combina las cabezas: Los resultados de cada cabeza se combinan de nuevo en un único tensor mediante el método combine_heads.
5. Aplica la transformación de salida: Por último, el tensor combinado se hace pasar por una transformación lineal de salida.

En resumen, la clase MultiHeadAttention encapsula el mecanismo de atención multicabezal utilizado habitualmente en los modelos de transformador. Se encarga de dividir la entrada en varias cabezas de atención, aplicar la atención a cada cabeza y luego combinar los resultados. Al hacerlo, el modelo puede captar diversas relaciones en los datos de entrada a diferentes escalas, mejorando la capacidad expresiva del modelo.

#### Redes feed-forward en función de la posición

In [3]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


##### Definición de la clase:
```python
class PositionwiseFeedForward(nn.Module):
```
La clase PositionwiseFeedForward es una subclase de nn.Module de PyTorch, lo que significa que heredará todas las funcionalidades necesarias para trabajar con capas de redes neuronales.

##### Inicialización:
```python
def __init__(self, d_model, d_ff, dropout=0.1):
    super(PositionWiseFeedForward, self).__init__()
    self.fc1 = nn.Linear(d_model, d_ff)
    self.fc2 = nn.Linear(d_ff, d_model)
    self.relu = nn.ReLU()
```

1. d_model: Dimensionalidad de la entrada y la salida del modelo.
2. d_ff: Dimensionalidad de la capa interna en la red feed-forward.
3. self.fc1 y self.fc2: Dos capas totales conectadas (lineales) con las dimensiones especificadas en d_model y d_ff.
4. self.relu: Función de activación ReLU (Unidad Lineal Rectificada), que introduce la no linealidad entre las dos capas lineales.

##### Método de avance (forward):
```python
def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x
```
```python
def forward(self, x):
    return self.fc2(self.relu(self.fc1(x)))
```
1. x: La entrada a la red feed-forward, que se espera que tenga la forma (batch_size, seq_length, d_model).
2. self.fc1(x): La entrada se pasa a través de la primera capa lineal(fc1).
3. self.relu(x): A continuación, la salida de fc1 se pasa por una función de activación ReLU. ReLU sustituye todos los valores negativos por ceros, intorduciendo la no linealidad en el modelo.
4. self.fc2(x): A continuación, la salida activada pasa por la segunda capa lineal (fc2), produciendo la salida final.

En resumen, la clase PositionWiseFeedForward define una red neuronal feed-forward en función de la posición que consta de dos capas lineales con una función de activación ReLU entre ellas. En el contexto de los modelos de transformador, esta red feed-forward se aplica a cada posición por separado y de forma idéntica. Ayuda a transformar los rasgos aprendidos por los mecanismos de atención dentro del transformador, actuando como un paso de procesamiento adicional para los resultados de la atención.

#### Codificación posicional

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


##### Definición de la clase:
```python
class PositionalEncoding(nn.Module):
```
La clase se define como una subclase de nn.Module de PyTorch, lo que permite utilizarla como una capa estándar de PyTorch.
##### Inicialización:
```python
def __init__(self, d_model, max_seq_length):
    super(PositionalEncoding, self).__init__()
    
    pe = torch.zeros(max_seq_length, d_model)
    position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    
    self.register_buffer('pe', pe.unsqueeze(0))
```

1. d_model: La dimensión de la entrada del modelo.
2. max_seq_length: La longitud máxima de la secuencia para la que se precalculan las codificaciones posicionales.
3. pe: Un tensor relleno de ceros, que se rellenará con codificaciones posicionales.
4. position: Un tensor que contiene los índices de posición de cada posición de la secuencia.
5. div_term: Término utilizado para escalar los índices de posición de una forma determinada.
6. La función seno se aplica a los índices pares y la función coseno a los impares de pe.
7. Por último, pe se registra como un búfer, lo que significa que formará parte del estado del módulo pero no se considerará un parámetro entrenable.

##### Método de avance (forward):
```python
def forward(self, x):
    return x + self.pe[:, :x.size(1)]
```
El método de avance simplemente añade las codificaciones posicionales a la entrada x.

Utiliza los primeros x.size(1) elementos de pe para garantizar que las codificaciones posicionales coinciden con la longitud de secuencia real de x.

Resumen

La clase PositionalEncoding añade información sobre la posición de las fichas dentro de la secuencia. Como el modelo del transformador carece de conocimiento inherente del orden de las fichas (debido a su mecanismo de autoatención), esta clase ayuda al modelo a considerar la posición de las fichas en la secuencia. Las funciones sinusoidales utilizadas se eligen para que el modelo aprenda fácilmente a atender a posiciones relativas, ya que producen una codificación única y suave para cada posición de la secuencia.



### 3. Construir los bloques Codificadores

In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

##### Definición de la clase:
```python
class PositionalEncoding(nn.Module):
```

La clase se define como una subclase de nn.Module de PyTorch, lo que significa que puede utilizarse como bloque de construcción de redes neuronales en PyTorch.

##### Inicialización:
```python
def __init__(self, d_model, num_heads, d_ff, dropout):
    super(EncoderLayer, self).__init__()
    self.self_attn = MultiHeadAttention(d_model, num_heads)
    self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)
```

##### Parámetros:
1. d_model: Dimensionalidad de la entrada.
2. num_heads: Número de cabezas de atención.
3. d_ff: Dimensionalidad de la capa interna en la red feed-forward en función de la posición.
4. dropout: Tasa de abandono para la regularización.


##### Componentes:
1. self.self_attn: Instancia de la clase MultiHeadAttention, que implementa el mecanismo de atención multicabezal.
2. self.feed_forward: Instancia de la clase PositionWiseFeedForward, que implementa la red feed-forward en función de la posición.
3. self.norm1 y self.norm2: Normalización de capas para estabilizar el entrenamiento.
4. self.dropout: Capa de abandono para la regularización.

##### Método de avance (forward):
```python
def forward(self, x, mask):
    attn_output = self.self_attn(x, x, x, mask)
    x = self.norm1(x + self.dropout(attn_output))
    ff_output = self.feed_forward(x)
    x = self.norm2(x + self.dropout(ff_output))
    return x
```
##### Entrada:
1. x: La entrada a la capa codificadora, que se espera que tenga la forma (batch_size, seq_length, d_model).
2. mask: Máscara opcional para enmascarar ciertas posiciones en la entrada.

##### Pasos de procesamiento:

1. Autoatención: La entrada x pasa a través del mecanismo de autoatención multicabezal.
2. Añade y normaliza (después de la atención): La salida de la atención se añade a la entrada original (conexión residual), seguida de un abandono y normalización mediante norm1.
3. Red feed-forward: La salida del paso anterior se hace pasar por la red feed-forward en función de la posición.
4. Añade y normaliza (después del avance): De forma similar al paso 2, la salida feed-forward se añade a la entrada de esta etapa (conexión residual), seguida de un abandono y normalización mediante norm2.
5. Salida: El tensor procesado se devuelve como salida de la capa codificadora.

##### Resumen:

La clase EncoderLayer define una sola capa del codificador del transformador. Encierra un mecanismo de autoatención multicabezal seguido de la red neuronal feed-forward en función de la posición, con conexiones residuales, normalización de capas y abandono aplicados según proceda. Juntos, estos componentes permiten al codificador captar relaciones complejas en los datos de entrada y transformarlas en una representación útil para las tareas posteriores. Normalmente, se apilan varias capas codificadoras de este tipo para formar la parte codificadora completa de un modelo de transformador.

### 4. Construir los bloques Decodificadores

In [6]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x


##### Definición de la clase:
```python
class DecoderLayer(nn.Module):
```
La clase PositionalEncoding es una subclase de nn.Module de PyTorch, lo que significa que heredará todas las funcionalidades necesarias para trabajar con capas de redes neuronales.
##### Inicialización:
```python
def __init__(self, d_model, num_heads, d_ff, dropout):
    super(DecoderLayer, self).__init__()
    self.self_attn = MultiHeadAttention(d_model, num_heads)
    self.cross_attn = MultiHeadAttention(d_model, num_heads)
    self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.norm3 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)
```
##### Parámetros:
1. d_model: Dimensionalidad de la entrada.
2. num_heads: Número de cabezas de atención.
3. d_ff: Dimensionalidad de la capa interna en la red feed-forward.
4. dropout: Tasa de abandono para la regularización.
##### Componentes:
1. self.self_attn: Instancia de la clase MultiHeadAttention, que implementa el mecanismo de autoatención multicabezal.
2. self.cross_attn: Mecanismo de atención multicabezal que atiende a la salida del codificador.
3. self.feed_forward: Instancia de la clase PositionWiseFeedForward, que implementa la red feed-forward en función de la posición.
4. self.norm1, self.norm2 y self.norm3: Normalización de capas para estabilizar el entrenamiento.
5. self.dropout: Capa de abandono para la regularización.
##### Método de avance (forward):
```python
def forward(self, x, enc_output, src_mask, tgt_mask):
    attn_output = self.self_attn(x, x, x, tgt_mask)
    x = self.norm1(x + self.dropout(attn_output))
    attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
    x = self.norm2(x + self.dropout(attn_output))
    ff_output = self.feed_forward(x)
    x = self.norm3(x + self.dropout(ff_output))
    return x
```

##### Entrada:

1. x: La entrada a la capa decodificadora.
2. enc_output: La salida del codificador correspondiente (utilizada en el paso de atención cruzada).
3. src_mask: Máscara de fuente para ignorar ciertas partes de la salida del codificador.
4. tgt_mask: Máscara de destino para ignorar determinadas partes de la entrada del descodificador.

##### Pasos de procesamiento:

1. Autoatención en la secuencia objetivo: La entrada x se procesa mediante un mecanismo de autoatención.
2. Añade y normaliza (tras la autoatención): El resultado de la autoatención se añade a la x original, seguido de un abandono y una normalización mediante norm1.
3. Atención cruzada con la salida del codificador: La salida normalizada del paso anterior se procesa mediante un mecanismo de atención cruzada que atiende a la salida enc_output del codificador.
4. Suma y normaliza (después de la atención cruzada): La salida de la atención cruzada se añade a la entrada de esta etapa, seguida de la eliminación y normalización mediante norm2.
5. Red feed-forward: La salida del paso anterior se pasa a través de la red feed-forward.
6. Añade y normaliza (después del avance): La salida feed-forward se añade a la entrada de esta etapa, seguida de un dropout y una normalización mediante norm3.
7. Salida: El tensor procesado se devuelve como salida de la capa decodificadora.

##### Resumen:

La clase DecoderLayer define una sola capa del descodificador del transformador. Consta de un mecanismo de autoatención multicabezal, un mecanismo de atención cruzada multicabezal (que atiende a la salida del codificador), una red neuronal feed-forward en función de la posición, y las correspondientes conexiones residuales, normalización de capas y capas de abandono. Esta combinación permite al descodificador generar salidas significativas basadas en las representaciones del codificador, teniendo en cuenta tanto la secuencia objetivo como la secuencia fuente. Al igual que ocurre con el codificador, se suelen apilar varias capas de decodificador para formar la parte decodificadora completa de un modelo de transformador.

A continuación, los bloques Codificador y Decodificador se combinan para construir el modelo completo del Transformador.

### 5. Combinar las capas codificadora y decodificadora para crear la red Transformer completa

In [7]:
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model,
        num_heads,
        num_layers,
        d_ff,
        max_seq_length,
        dropout,
    ):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.decoder_layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (
            1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)
        ).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(
            self.positional_encoding(self.encoder_embedding(src))
        )
        tgt_embedded = self.dropout(
            self.positional_encoding(self.decoder_embedding(tgt))
        )

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output


##### Definición de clase:
```python
class Transformer(nn.Module):
```
La clase Transformer es una subclase de nn.Module de PyTorch, lo que significa que heredará todas las funcionalidades necesarias para trabajar con capas de redes neuronales.

#### Inicialización:
```python
def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
```

##### Parámetros:

1. src_vocab_size: Tamaño del vocabulario fuente.
2. tgt_vocab_size: Tamaño del vocabulario objetivo.
3. d_model: La dimensionalidad de las incrustaciones del modelo.
4. num_heads: Número de cabezas de atención en el mecanismo de atención multicabeza.
5. num_layers: Número de capas tanto para el codificador como para el descodificador.
6. d_ff: Dimensionalidad de la capa interna en la red feed-forward.
7. max_seq_length: Longitud máxima de la secuencia para la codificación posicional.
8. dropout: Tasa de abandono por regularización.

#### Componentes:

1. self.encoder_embedding: Capa de incrustación de la secuencia fuente.
2. self.decoder_embedding: Capa de incrustación de la secuencia objetivo.
3. self.positional_encoding: Componente de codificación posicional.
4. self.encoder_layers: Una lista de capas del codificador.
5. self.decoder_layers: Una lista de las capas del descodificador.
6. self.fc: Asignación de la capa final totalmente conectada (lineal) al tamaño del vocabulario objetivo.
7. self.dropout: Capa de abandono.

#### Generar método de máscara:
```python
def generate_mask(self, src, tgt):
```

Este método se utiliza para crear máscaras para las secuencias de origen y de destino, garantizando que se ignoren las fichas de relleno y que las fichas futuras no sean visibles durante el entrenamiento para la secuencia de destino.

##### Método de avance:
```python
def forward(self, src, tgt):
```

El método de avance define el paso hacia delante del Transformador, tomando las secuencias origen y destino y produciendo las predicciones de salida.

1. Incrustación de entrada y codificación posicional: Las secuencias de origen y destino se incrustan primero utilizando sus respectivas capas de incrustación y luego se añaden a sus codificaciones posicionales.
2. Capas del codificador: La secuencia fuente pasa por las capas del codificador, y la salida final del codificador representa la secuencia fuente procesada.
3. Capas de descodificación: La secuencia objetivo y la salida del codificador pasan por las capas del descodificador, dando como resultado la salida del descodificador.
4. Capa lineal final: La salida del descodificador se asigna al tamaño del vocabulario objetivo mediante una capa totalmente conectada (lineal).

##### Salida:

El resultado final es un tensor que representa las predicciones del modelo para la secuencia objetivo.

##### Resumen:

La clase Transformador reúne los distintos componentes de un modelo Transformador, incluidas las incrustaciones, la codificación posicional, las capas codificadoras y las capas decodificadoras. Proporciona una interfaz cómoda para el entrenamiento y la inferencia, encapsulando las complejidades de la atención multicabezal, las redes feed-forward y la normalización de capas.

Esta implementación sigue la arquitectura estándar de Transformer, lo que la hace adecuada para tareas de secuencia a secuencia como la traducción automática, el resumen de textos, etc. Incluir el enmascaramiento garantiza que el modelo se ciña a las dependencias causales dentro de las secuencias, ignorando las fichas de relleno y evitando la fuga de información de las fichas futuras.

Estos pasos secuenciales capacitan al modelo Transformador para procesar eficazmente las secuencias de entrada y producir las secuencias de salida correspondientes.

### Entrenamiento del modelo de transformador PyTorch
#### Preparación de los datos de la muestra
A efectos ilustrativos, en este ejemplo se elaborará un conjunto de datos ficticio. Sin embargo, en un escenario práctico, se emplearía un conjunto de datos más sustancial, y el proceso implicaría el preprocesamiento del texto junto con la creación de mapeos de vocabulario tanto para la lengua de origen como para la de destino.

In [8]:
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1

transformer = Transformer(
    src_vocab_size,
    tgt_vocab_size,
    d_model,
    num_heads,
    num_layers,
    d_ff,
    max_seq_length,
    dropout,
)

# Generate random sample data
src_data = torch.randint(
    1, src_vocab_size, (64, max_seq_length)
)  # (batch_size, seq_length)
tgt_data = torch.randint(
    1, tgt_vocab_size, (64, max_seq_length)
)  # (batch_size, seq_length)


##### Hiperparámetros:

Estos valores definen la arquitectura y el comportamiento del modelo de transformador:

1. src_vocab_size, tgt_vocab_size: Tamaño de los vocabularios de las secuencias origen y destino, ambos fijados en 5000.
2. d_model: Dimensionalidad de las incrustaciones del modelo, fijada en 512.
3. num_heads: Número de cabezales de atención en el mecanismo de atención multicabezal, fijado en 8.
4. num_layers: Número de capas tanto para el codificador como para el descodificador, fijado en 6.
5. d_ff: Dimensionalidad de la capa interna de la red feed-forward, fijada en 2048.
6. max_seq_length: Longitud máxima de la secuencia para la codificación posicional, fijada en 100.
7. dropout: Tasa de abandono para la regularización, fijada en 0,1.
Como referencia, la tabla siguiente describe los hiperparámetros más comunes de los modelos Transformer y sus valores:

| Hiperparámetro        | Valores típicos          | Impacto en el rendimiento                                      |
|----------------------|------------------------|--------------------------------------------------------------|
| d_model             | 256, 512, 1024         | Los valores más altos aumentan la capacidad del modelo, pero requieren más cálculo |
| num_heads           | 8, 12, 16              | Más cabezales pueden captar diversos aspectos de los datos, pero son computacionalmente intensivos |
| número_capas        | 6, 12, 24              | Más capas mejoran el poder de representación, pero pueden llevar a un sobreajuste |
| d_ff                | 2048, 4096             | Las redes feed-forward más grandes aumentan la robustez del modelo |
| abandono            | 0.1, 0.3               | Regulariza el modelo para evitar el sobreajuste              |
| ritmo de aprendizaje | 0.0001 - 0.001         | Afecta a la velocidad de convergencia y a la estabilidad     |
| tamaño del lote     | 32, 64, 128            | Los lotes de mayor tamaño mejoran la estabilidad del aprendizaje, pero requieren más memoria |

##### Crear una instancia de Transformador:
```python
transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)
```
Esta línea crea una instancia de la clase Transformer, inicializándola con los hiperparámetros dados. La instancia tendrá la arquitectura y el comportamiento definidos por estos hiperparámetros.

##### Generar datos de muestras aleatorias:

Las siguientes líneas generan secuencias aleatorias de origen y destino:

1. src_data: Números enteros aleatorios entre 1 y src_vocab_size, que representan un lote de secuencias fuente con forma (64, max_seq_length).
2. tgt_data: Números enteros aleatorios entre 1 y tgt_vocab_size, que representan un lote de secuencias objetivo con forma (64, max_seq_length).
3. Estas secuencias aleatorias pueden utilizarse como entradas del modelo de transformador, simulando un lote de datos con 64 ejemplos y secuencias de longitud 100.

##### Resumen:

El fragmento de código muestra cómo inicializar un modelo Transformer y generar secuencias aleatorias de origen y destino que se pueden introducir en él. Los hiperparámetros elegidos determinan la estructura y las propiedades específicas del Transformador. Esta configuración podría formar parte de un guión más amplio en el que el modelo se entrenara y evaluara en tareas reales de secuencia a secuencia, como la traducción automática o el resumen de textos.


### Entrenamiento del modelo
A continuación, se entrenará el modelo utilizando los datos de muestra antes mencionados. Sin embargo, en un escenario real, se emplearía un conjunto de datos significativamente mayor, que normalmente se dividiría en conjuntos distintos con fines de entrenamiento y validación.

In [9]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer.train()

for epoch in range(100):
    optimizer.zero_grad()
    output = transformer(src_data, tgt_data[:, :-1])
    loss = criterion(
        output.contiguous().view(-1, tgt_vocab_size),
        tgt_data[:, 1:].contiguous().view(-1),
    )
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")
# Save the model
torch.save(transformer.state_dict(), "transformer_model.pth")

Epoch: 1, Loss: 8.68265438079834
Epoch: 2, Loss: 8.55888843536377
Epoch: 3, Loss: 8.486761093139648
Epoch: 4, Loss: 8.43582534790039


KeyboardInterrupt: 

##### Función de pérdida y optimizador:

1. criterion = nn.CrossEntropyLoss(ignore_index=0): Define la función de pérdida como pérdida de entropía cruzada. El argumento ignore_index se establece en 0, lo que significa que la pérdida no tendrá en cuenta los objetivos con un índice 0 (normalmente reservado para los tokens de relleno).
2. optimizer = optim.Adam(...): Define el optimizador como Adam con una tasa de aprendizaje de 0,0001 y valores beta específicos.

##### Modo de entrenamiento del modelo:
```python
transformer.train(): Establece el modelo del transformador en modo de entrenamiento, permitiendo comportamientos como el abandono que sólo se aplican durante el entrenamiento.
```
##### Bucle de entrenamiento:

El fragmento de código entrena el modelo durante 100 épocas utilizando un bucle de entrenamiento típico:

1. for epoch in range(100): Itera más de 100 épocas de entrenamiento.
2. optimizer.zero_grad(): Borra los degradados de la iteración anterior.
3. output = transformer(src_data, tgt_data[:, :-1]): Pasa los datos de origen y los datos de destino (excluyendo el último token de cada secuencia) a través del transformador. Esto es habitual en tareas de secuencia a secuencia en las que el objetivo se desplaza un token.
4. loss = criterion(...): Calcula la pérdida entre las predicciones del modelo y los datos objetivo (excluyendo el primer token de cada secuencia). La pérdida se calcula remodelando los datos en tensores unidimensionales y utilizando la función de pérdida de entropía cruzada.
5. loss.backward(): Calcula los gradientes de la pérdida con respecto a los parámetros del modelo.
6. optimizer.step(): Actualiza los parámetros del modelo utilizando los gradientes calculados.
7. print(f"Epoch: {epoch+1}, Loss: {loss.item()}"): Imprime el número de época actual y el valor de pérdida de esa época.

##### Resumen:

Este fragmento de código entrena el modelo transformador con secuencias de origen y destino generadas aleatoriamente durante 100 épocas. Utiliza el optimizador Adam y la función de pérdida de entropía cruzada. La pérdida se imprime para cada época, lo que te permite controlar el progreso del entrenamiento. En un escenario real, sustituirías las secuencias aleatorias de origen y destino por datos reales de tu tarea, como la traducción automática.

### Evaluación del rendimiento del modelo de transformador
Tras entrenar el modelo, se puede evaluar su rendimiento en un conjunto de datos de validación o de prueba. El siguiente es un ejemplo de cómo podría hacerse:

In [10]:
transformer.eval()

# Generate random sample validation data
val_src_data = torch.randint(
    1, src_vocab_size, (64, max_seq_length)
)  # (batch_size, seq_length)
val_tgt_data = torch.randint(
    1, tgt_vocab_size, (64, max_seq_length)
)  # (batch_size, seq_length)

with torch.no_grad():
    val_output = transformer(val_src_data, val_tgt_data[:, :-1])
    val_loss = criterion(
        val_output.contiguous().view(-1, tgt_vocab_size),
        val_tgt_data[:, 1:].contiguous().view(-1),
    )
    print(f"Validation Loss: {val_loss.item()}")


Validation Loss: 8.673807144165039


Modo de evaluación:

transformer.eval(): Pone el modelo de transformador en modo de evaluación. Esto es importante porque desactiva ciertos comportamientos, como el abandono, que sólo se utilizan durante el entrenamiento.
Genera datos de validación aleatorios:

val_src_data: Números enteros aleatorios entre 1 y src_vocab_size, que representan un lote de secuencias fuente de validación con forma (64, max_seq_length).
val_tgt_data: Números enteros aleatorios entre 1 y tgt_vocab_size, que representan un lote de secuencias objetivo de validación con forma (64, max_seq_length).
Bucle de validación:

with torch.no_grad(): Desactiva el cálculo de gradientes, ya que no necesitamos calcular gradientes durante la validación. Esto puede reducir el consumo de memoria y acelerar los cálculos.
val_output = transformer(val_src_data, val_tgt_data[:, :-1]): Pasa los datos de origen de la validación y los datos de destino de la validación (excluyendo el último token de cada secuencia) a través del transformador.
val_loss = criterion(...): Calcula la pérdida entre las predicciones del modelo y los datos objetivo de validación (excluyendo el primer token de cada secuencia). La pérdida se calcula remodelando los datos en tensores unidimensionales y utilizando la función de pérdida de entropía cruzada definida anteriormente.
print(f"Validation Loss: {val_loss.item()}"): Imprime el valor de pérdida de validación.
Resumen:

Este fragmento de código evalúa el modelo transformador en un conjunto de datos de validación generado aleatoriamente, calcula la pérdida de validación y la imprime. En un escenario real, los datos de validación aleatorios deberían sustituirse por datos de validación reales de la tarea en la que estás trabajando. La pérdida de validación puede darte una indicación de lo bien que funciona tu modelo con datos no vistos, que es una medida crítica de la capacidad de generalización del modelo.

Para más detalles sobre los Transformadores y la Cara Abrazada, es útil nuestro tutorial Introducción al uso de los Transformadores y la Cara Abrazada.

### Conclusión y otros recursos
En conclusión, este tutorial ha demostrado cómo construir un modelo Transformer utilizando PyTorch, una de las herramientas más versátiles para el aprendizaje profundo. Con su capacidad de paralelización y de captar dependencias a largo plazo en los datos, los Transformadores tienen un inmenso potencial en diversos campos, especialmente en tareas de PNL como la traducción, el resumen y el análisis de sentimientos.

Para quienes deseen profundizar en los conceptos y técnicas avanzados del aprendizaje profundo, considera la posibilidad de explorar el curso Aprendizaje profundo avanzado con Keras en DataCamp. También puedes leer sobre la construcción de una red neuronal sencilla con PyTorch en otro tutorial.

Para entrenar el modelo **Transformer** implementado en la página web visible con archivos de texto como `el_quijote.txt`, sigue estos pasos:



### **1. Cargar el archivo de texto**
Primero, lee el contenido del archivo y conviértelo en datos utilizables:

In [32]:
with open("el_quijote.txt", "r", encoding="utf-8") as f:
    text = f.read()

### **2. Tokenizar el texto**
Dado que el modelo de la página usa PyTorch puro, puedes usar `torchtext` para tokenizar el texto:

In [34]:
from torchtext.data.utils import get_tokenizer

tokenizer = get_tokenizer("basic_english")
tokens = tokenizer(text)

ModuleNotFoundError: No module named 'torchtext'

### **3. Crear un vocabulario**
Construye un vocabulario a partir de los tokens:

In [ ]:
from collections import Counter
from torchtext.vocab import Vocab

counter = Counter(tokens)
vocab = Vocab(counter, specials=["<pad>", "<unk>", "<bos>", "<eos>"])


### **4. Convertir texto en tensores**
Convierte los tokens en índices numéricos:



In [ ]:
def numericalize(text, vocab):
    return [vocab[token] for token in text]

numericalized_text = numericalize(tokens, vocab)

### **5. Crear un conjunto de datos y DataLoader**
Divide el texto en secuencias y crea un `DataLoader`:

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = [data[i:i+seq_length] for i in range(0, len(data)-seq_length, seq_length)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.long)

seq_length = 50  # Ajusta según necesidad
dataset = TextDataset(numericalized_text, seq_length)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

### **6. Ajustar el modelo de la página**
Si el modelo de la página ya está implementado, simplemente usa su instancia y define el entrenamiento:

In [ ]:
import torch.nn as nn
import torch.optim as optim

model = Transformer(...)  # Usa la implementación de la página
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

def train(model, dataloader, optimizer, criterion, epochs=3):
    model.train()
    for epoch in range(epochs):
        for batch in dataloader:
            optimizer.zero_grad()
            output = model(batch)  # Ajusta según la entrada esperada
            loss = criterion(output.view(-1, output.size(-1)), batch.view(-1))  # Ajusta según la tarea
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item()}")

train(model, dataloader, optimizer, criterion)


### **7. Guardar el modelo entrenado**
Después del entrenamiento, guarda el modelo para futuras inferencias:


torch.save(model.state_dict(), "transformer_model.pth")

Este flujo te permitirá entrenar el modelo **Transformer** de la página con archivos de texto como `el_quijote.txt`. Si necesitas más detalles sobre el preprocesamiento o ajustes específicos, dime y te ayudo. 😊

También puedes consultar [este recurso](https://huggingface.co/docs/transformers/v4.48.0/es/run_scripts) sobre entrenamiento de modelos Transformer en PyTorch. ¡Buena suerte con tu entrenamiento! 🚀

Para generar texto con el modelo **Transformer** entrenado en PyTorch con los archivos de texto como `el_quijote.txt`, sigue estos pasos:

### **1. Cargar el modelo entrenado**
Si ya has guardado el modelo después del entrenamiento, primero cárgalo:

In [ ]:
import torch

model = Transformer(...)  # Usa la implementación de la página
model.load_state_dict(torch.load("transformer_model.pth"))
model.eval()  # Cambia a modo de evaluación

### **2. Preparar el texto inicial**
Define una frase inicial para que el modelo genere texto basado en lo aprendido:

In [ ]:
start_text = "En un lugar de la Mancha"

### **3. Tokenizar la entrada**
Convierte el texto en tokens numéricos usando el mismo vocabulario que usaste en el entrenamiento:

In [ ]:
tokens = tokenizer(start_text)
numericalized = torch.tensor([vocab[token] for token in tokens], dtype=torch.long).unsqueeze(0)

### **4. Generar texto paso a paso**
Usa el modelo para generar nuevas palabras de manera autoregresiva:

In [ ]:
def generate_text(model, input_tensor, max_length=100):
    model.eval()
    generated = input_tensor

    for _ in range(max_length):
        output = model(generated)  # Ajusta según la implementación del modelo
        next_token = torch.argmax(output[:, -1, :], dim=-1)  # Selecciona el token más probable
        generated = torch.cat((generated, next_token.unsqueeze(0)), dim=1)

        if next_token.item() == vocab["<eos>"]:  # Detener si se genera un token de fin de secuencia
            break

    return " ".join([vocab.lookup_token(idx) for idx in generated.squeeze().tolist()])

generated_text = generate_text(model, numericalized)
print(generated_text)

### **5. Ajustar hiperparámetros**
Puedes mejorar la generación de texto ajustando:
- **Temperatura**: Controla la aleatoriedad en la selección de palabras.
- **Top-k sampling**: Limita la selección a los `k` tokens más probables.
- **Top-p (nucleus) sampling**: Ajusta la probabilidad acumulada para elegir tokens.



Para mejorar la generación de texto con tu modelo **Transformer en PyTorch**, puedes aplicar **temperatura**, **top-k sampling** y **top-p (nucleus) sampling** para controlar la aleatoriedad y creatividad del texto generado.

### **1. Implementar generación de texto con temperatura**
La **temperatura** ajusta la distribución de probabilidades para hacer el texto más creativo o más conservador:
```python
import torch.nn.functional as F

def apply_temperature(logits, temperature=1.0):
    logits = logits / temperature
    return F.softmax(logits, dim=-1)
```
- **Temperatura baja (<1)**: Hace el modelo más conservador, elige palabras con mayor probabilidad.
- **Temperatura alta (>1)**: Hace el modelo más creativo, permitiendo palabras menos probables.

---

### **2. Implementar Top-k Sampling**
El **top-k sampling** limita la selección de palabras a las `k` más probables:
```python
def top_k_sampling(logits, k=50):
    values, indices = torch.topk(logits, k)
    probs = F.softmax(values, dim=-1)
    return indices[torch.multinomial(probs, 1)]
```
- **k pequeño (5-20)**: Genera texto más coherente y enfocado.
- **k grande (50-100)**: Introduce más variedad y creatividad.

---

### **3. Implementar Top-p (Nucleus) Sampling**
El **top-p sampling** selecciona palabras hasta que la suma de sus probabilidades alcanza un umbral `p`:
```python
def top_p_sampling(logits, p=0.9):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

    filtered_indices = sorted_indices[cumulative_probs <= p]
    filtered_probs = F.softmax(sorted_logits[cumulative_probs <= p], dim=-1)

    return filtered_indices[torch.multinomial(filtered_probs, 1)]
```
- **p bajo (0.7-0.8)**: Genera texto más preciso y estructurado.
- **p alto (0.9-1.0)**: Permite más creatividad y diversidad.

---

### **4. Integrar en la generación de texto**
```python
def generate_text(model, input_tensor, max_length=100, temperature=1.0, top_k=50, top_p=0.9):
    model.eval()
    generated = input_tensor

    for _ in range(max_length):
        output = model(generated)
        logits = output[:, -1, :]

        logits = apply_temperature(logits, temperature)

        if top_p < 1.0:
            next_token = top_p_sampling(logits, top_p)
        elif top_k > 0:
            next_token = top_k_sampling(logits, top_k)
        else:
            next_token = torch.argmax(logits, dim=-1)

        generated = torch.cat((generated, next_token.unsqueeze(0)), dim=1)

        if next_token.item() == vocab["<eos>"]:
            break

    return " ".join([vocab.lookup_token(idx) for idx in generated.squeeze().tolist()])
```

---

### **5. Ejecutar la generación de texto**
```python
generated_text = generate_text(model, numericalized, temperature=0.8, top_k=50, top_p=0.9)
print(generated_text)
```

Este código te permitirá generar texto con mayor control sobre la creatividad y coherencia. ¿Quieres ajustar algún parámetro o probar otra estrategia? 🚀